# Conclusiones, limitaciones y trabajo futuro

En este capítulo cerramos el estudio: qué pondríamos en producción y bajo qué supuestos, qué hipótesis se sostienen, qué amenazas quedan abiertas y qué experimento las reduciría, y qué objeciones haría un revisor externo.

In [1]:
import pandas as pd
from IPython.display import Markdown, display

from project.book_utils import ci, exists, need, style

style()
pd.set_option("display.precision", 3)
md = lambda t: display(Markdown(t))  # noqa: E731
SG = need("analysis/seg_stats/stats.pkl")
NL = need("analysis/nlp_stats/stats.pkl")
LS, LN = SG["labels"], NL["labels"]

## Hipótesis

In [2]:
rows = []
h1 = SG.get("h1", {})
if "25 %" in h1:
    a, b = h1["25 %"], h1.get("100 %")
    ref = h1.get("parte1_refutada", False) or h1.get("parte2_refutada", False)
    rows.append(dict(hipótesis="H1: el preentrenamiento COCO ayuda más con pocos datos",
                     evidencia=f"Mask R-CNN − U-Net++ al 25 %: {ci(a['diff'], a['lo'], a['hi'])}"
                     + (f"; al 100 %: {ci(b['diff'], b['lo'], b['hi'])}" if b else ""),
                     veredicto="refutada" if ref else "no refutada"))
else:
    rows.append(dict(hipótesis="H1: el preentrenamiento COCO ayuda más con pocos datos", evidencia="faltan los entrenamientos al 25 %",
                     veredicto="sin contrastar"))
h2 = SG.get("h2", {})
if h2.get("spearman_auc"):
    rows.append(dict(hipótesis="H2: las métricas de píxel saturan y no ordenan como AP",
                     evidencia=f"ρ de Spearman (AUC-ROC de píxel, AP) = {h2['spearman_auc'][0]:.2f} con {h2['n_models']} modelos",
                     veredicto="refutada" if h2["refuted"] else "no refutada"))
h3 = SG["stratified"]["h3"]
rows.append(dict(hipótesis="H3: el orden de modelos cambia por tipo celular", evidencia=f"W de Kendall = {h3['kendall_w']:.2f}",
                 veredicto="refutada" if h3["identical"] else "no refutada"))
h4 = NL.get("h4", {})
if h4:
    rows.append(dict(hipótesis="H4: DistilBERT y TF-IDF + XGBoost no difieren significativamente",
                     evidencia=f"ΔF1 = {ci(h4['diff_bert_minus_xgb'], h4['lo'], h4['hi'])}; p Holm McNemar {h4['p_mcnemar_holm']:.3g}",
                     veredicto=("refutada" if h4["refuted"] else "no refutada") + ("; equivalencia práctica" if h4["equivalent"] else "")))
h5 = NL["calibration"]["h5"]
if h5:
    rows.append(dict(hipótesis="H5: DistilBERT está mal calibrado y el escalado de temperatura lo corrige",
                     evidencia=f"ECE {h5['ece_bert']:.3f} → {h5['ece_bert_ts']:.3f}; reducción {ci(h5['reduction'], h5['red_lo'], h5['red_hi'])}",
                     veredicto="refutada" if h5["refuted"] else "no refutada"))
display(pd.DataFrame(rows).style.hide(axis="index"))

hipótesis,evidencia,veredicto
H1: el preentrenamiento COCO ayuda más con pocos datos,"Mask R-CNN − U-Net++ al 25 %: -0.091 [-0.105, -0.078]; al 100 %: 0.014 [0.007, 0.021]",refutada
H2: las métricas de píxel saturan y no ordenan como AP,"ρ de Spearman (AUC-ROC de píxel, AP) = 0.75 con 7 modelos",no refutada
H3: el orden de modelos cambia por tipo celular,W de Kendall = 0.82,no refutada
H4: DistilBERT y TF-IDF + XGBoost no difieren significativamente,"ΔF1 = 0.039 [-0.002, 0.086]; p Holm McNemar 0.02",refutada
H5: DistilBERT está mal calibrado y el escalado de temperatura lo corrige,"ECE 0.088 → 0.070; reducción 0.032 [0.000, 0.057]",no refutada


«No refutada» significa que el criterio de refutación fijado antes de entrenar no se cumplió; no equivale a una prueba de la hipótesis. Las hipótesis refutadas son resultados tan informativos como las otras: indican dónde la intuición de partida no se sostiene con estos datos.

## El resultado más inesperado

In [3]:
_fr = SG.get("ablations", {}).get("fracción")
_r25 = _fr["table"].set_index("variante").loc["25 %"] if _fr is not None and "25 %" in set(_fr["table"].variante) else None
if h1.get("parte1_sentido_contrario") and _r25 is not None:
    a, b = h1["25 %"], h1["100 %"]
    md(f"Esperábamos que el preentrenamiento en COCO pesara más cuanto menos datos hubiera. Ocurrió lo contrario: con el 25 % del "
       f"entrenamiento, Mask R-CNN preentrenado queda {abs(a['diff']):.3f} de AP por debajo de la U-Net++ entrenada desde cero "
       f"({ci(a['diff'], a['lo'], a['hi'])}), y solo con el 100 % lo supera ({ci(b['diff'], b['lo'], b['hi'])}). La U-Net++, además, "
       f"apenas nota la reducción de datos: la diferencia de AP entre el 25 % y el 100 % es {ci(round(_r25['diff'], 3) + 0.0, _r25['lo'], _r25['hi'])}"
       ".\n\n"
       "Dos mecanismos pueden explicarlo, y el experimento no los separa. El primero es el presupuesto de optimización: la U-Net++ "
       "hace el mismo número de pasos por época con cualquier fracción, mientras que la época de Mask R-CNN se acorta con los datos y "
       "su parada temprana actúa antes (tabla de pasos del capítulo {doc}`13_seg_resultados`). El segundo es la naturaleza de la "
       "tarea: una red que clasifica píxeles en recortes de 256 × 256 obtiene miles de ejemplos de borde e interior de cada imagen, "
       "mientras que un detector necesita ver muchas instancias completas para ajustar anclas, cajas y máscaras. Las dos cifras "
       "salen de una semilla por modelo, así que son exploratorias; el experimento que las separaría es repetir el 25 % con el "
       "mismo número de pasos de gradiente en las dos familias y con tres semillas.")
else:
    md("La sección se completa cuando están las corridas al 25 % y la evaluación de test (`python scripts/analysis.py seg`).")

Esperábamos que el preentrenamiento en COCO pesara más cuanto menos datos hubiera. Ocurrió lo contrario: con el 25 % del entrenamiento, Mask R-CNN preentrenado queda 0.091 de AP por debajo de la U-Net++ entrenada desde cero (-0.091 [-0.105, -0.078]), y solo con el 100 % lo supera (0.014 [0.007, 0.021]). La U-Net++, además, apenas nota la reducción de datos: la diferencia de AP entre el 25 % y el 100 % es 0.000 [-0.005, 0.005].

Dos mecanismos pueden explicarlo, y el experimento no los separa. El primero es el presupuesto de optimización: la U-Net++ hace el mismo número de pasos por época con cualquier fracción, mientras que la época de Mask R-CNN se acorta con los datos y su parada temprana actúa antes (tabla de pasos del capítulo {doc}`13_seg_resultados`). El segundo es la naturaleza de la tarea: una red que clasifica píxeles en recortes de 256 × 256 obtiene miles de ejemplos de borde e interior de cada imagen, mientras que un detector necesita ver muchas instancias completas para ajustar anclas, cajas y máscaras. Las dos cifras salen de una semilla por modelo, así que son exploratorias; el experimento que las separaría es repetir el 25 % con el mismo número de pasos de gradiente en las dos familias y con tres semillas.

## ¿Qué pondríamos en producción y cuándo dejaría de valer la decisión?

In [4]:
SUM = SG["summary"]
DEC = need("analysis/decision/decision.pkl")
best_seg = DEC["seg"]["winner"] or SUM.ap.idxmax()
T = NL["results"].drop("majority", errors="ignore")
served = need("mlops/export_text/export_text.pkl")["choice"] if exists("mlops/export_text/export_text.pkl") else T.f1_macro.idxmax()
md(f"**Segmentación: {LS.get(best_seg, best_seg)}** (AP en test {ci(SUM.ap[best_seg], SUM.ap_lo[best_seg], SUM.ap_hi[best_seg])}), "
   f"para conteo y morfometría con revisión de un experto en las imágenes de baja confianza. "
   f"**Texto: {LN.get(served, served)}** (F1 macro en test {T.f1_macro[served]:.3f} ± {T.f1_macro_sd[served]:.3f}), solo como sugerencia a una "
   f"persona, nunca para decidir sobre candidatos (capítulo {{doc}}`25_nlp_analisis_critico`).")
if DEC["seg"]["winner"] is not None:
    md(f"La elección de segmentación es la del análisis multicriterio del capítulo {{doc}}`40_comparacion_final`, que gana en el "
       f"{100 * DEC['seg']['dirichlet'].get(best_seg, 0):.0f} % de los vectores de pesos muestreados; la de texto es la de la regla de servicio "
       f"pre-registrada.")
if exists("mlops/export_seg/export_seg.pkl") and need("mlops/export_seg/export_seg.pkl")["model"] != best_seg:
    _srv = need("mlops/export_seg/export_seg.pkl")["model"]
    md(f"El servicio actual sirve {LS.get(_srv, _srv)} en segmentación; recomendamos sustituirlo por {LS.get(best_seg, best_seg)} cuando "
       "pase la misma batería de paridad, conjunto dorado y despliegue en sombra (capítulo {doc}`40_comparacion_final`).")

**Segmentación: Cellpose ajustado** (AP en test 0.253 [0.226, 0.281]), para conteo y morfometría con revisión de un experto en las imágenes de baja confianza. **Texto: DistilBERT** (F1 macro en test 0.774 ± 0.012), solo como sugerencia a una persona, nunca para decidir sobre candidatos (capítulo {doc}`25_nlp_analisis_critico`).

La elección de segmentación es la del análisis multicriterio del capítulo {doc}`40_comparacion_final`, que gana en el 100 % de los vectores de pesos muestreados; la de texto es la de la regla de servicio pre-registrada.

El servicio actual sirve U-Net++ en segmentación; recomendamos sustituirlo por Cellpose ajustado cuando pase la misma batería de paridad, conjunto dorado y despliegue en sombra (capítulo {doc}`40_comparacion_final`).

La decisión deja de valer si cambia alguno de estos supuestos:

- **Segmentación.** Mismo microscopio, modalidad (contraste de fase), resolución y protocolo de cultivo que Sartorius; mismas tres líneas celulares o una nueva evaluada con su propio conjunto etiquetado; densidades dentro del rango del entrenamiento. Una alarma sostenida de los detectores de imagen del capítulo {doc}`31_monitoreo_deriva` es la señal operativa de que alguno cambió.
- **Texto.** Currículos en inglés, con una estructura similar (encabezado, experiencia, educación, habilidades) y las mismas 24 categorías; una persona revisa cada sugerencia. Si el uso pasa a filtrar o descartar candidatos de forma automática, la decisión deja de valer por completo, con independencia de la métrica.

## Amenazas a la validez y experimentos que las reducirían

| Amenaza | Experimento que la reduciría |
|---|---|
| Un solo microscopio y laboratorio en segmentación | Evaluar sin reentrenar en otro conjunto de contraste de fase, como LIVECell {cite:p}`edlund2021livecell`, y medir la caída |
| Anotaciones con errores en astrocitos | Reanotar una muestra de test por dos expertos y recalcular el AP con la anotación corregida |
| Etiquetas de Jarvis = categoría de publicación | Etiquetar una muestra con la profesión verificada y medir el acuerdo con la etiqueta original |
| Equidad medida con un indicio débil | Auditar con atributos declarados voluntariamente por candidatos reales |
| Búsqueda de hiperparámetros desigual para DistilBERT | Darle el mismo presupuesto de 30 *trials* y validación anidada |
| Réplicas de ablaciones recortadas por presupuesto | Completar 3 semillas por variante y repetir el modelo mixto |
| H1 al 25 % mezcla preentrenamiento y presupuesto de optimización | Repetir el contraste con el mismo número de pasos de gradiente en las dos familias y tres semillas |
| Sesgo optimista de validación sin corregir en XGBoost, FastText y las redes de texto | Ejecutar la validación anidada que quedó fuera de la cola (10 a 15 h por modelo en CPU) |
| Anonimización incompleta del corpus de texto | Pasar un detector de entidades nombradas por todo el corpus y medir cuántos nombres propios quedan |
| Deriva de covariables del texto vigilada solo por la longitud con DistilBERT servido | Añadir un MMD sobre los *embeddings* del propio modelo servido y repetir el experimento de deriva |

## ¿Qué cambiaría con el doble o con la mitad del presupuesto?

**Con el doble.** Completaríamos las tres semillas de todas las ablaciones de segmentación, igualaríamos la búsqueda de DistilBERT con la de los modelos ligeros (30 *trials* con validación anidada), añadiríamos un modelo de texto con ventana larga para no truncar los currículos y evaluaríamos la segmentación en un conjunto externo.

**Con la mitad.** El presupuesto real ya obligó a recortar la validación anidada de XGBoost y FastText y parte de las semillas de las ablaciones (enmiendas del capítulo {doc}`03_declaracion_apriori`). Con la mitad mantendríamos las tres semillas de los modelos principales y el análisis estadístico, que son lo que sostiene las conclusiones, y recortaríamos además las ablaciones de fracción de datos, la variante Hover y, en texto, la CNN-1D. Las conclusiones sobre H1 y RQ2 quedarían como exploratorias.

## Reflexión ética y de impacto

**Usos indebidos posibles.** El clasificador de currículos puede usarse para filtrar candidatos sin revisión, o para inferir la profesión o el perfil de una persona a partir de un texto que no envió con ese fin. El segmentador puede usarse fuera de su dominio (por ejemplo, en imágenes clínicas) y producir medidas con apariencia de precisión que no la tienen.

**Poblaciones afectadas.** En texto, los candidatos cuyos currículos no siguen el formato mayoritario del conjunto (otro idioma, trayectorias mixtas, profesiones con pocos ejemplos como BPO) son los que más errores reciben. En segmentación, los usuarios afectados son los investigadores cuyas conclusiones dependen de conteos sesgados en las líneas celulares más difíciles.

**Salvaguardas.** Las del capítulo {doc}`25_nlp_analisis_critico` (decisión humana, derivación por confianza, auditoría de equidad con atributos declarados, monitoreo de deriva y transparencia), las tarjetas de modelo del capítulo {doc}`30_mlops` con usos fuera de alcance explícitos, y la regla de no servir un modelo cuya regresión sobre el conjunto dorado empeore.

## Autoevaluación: las cinco objeciones más serias de un revisor externo

In [5]:
obj = []
# 1. tamaño de test
n_img = int(SUM.n_images.max())
obj.append(("El test es pequeño: las diferencias entre modelos podrían ser ruido de muestreo.",
            f"Es una limitación real. Segmentación usa {n_img} imágenes de test y texto 373 documentos (≈ 15 por clase). Por eso reportamos IC "
            f"*bootstrap* en todas las cifras, corrección de Holm en todas las familias de comparaciones y, en texto, la *t* de Nadeau–Bengio "
            f"sobre 5 pliegues además de McNemar. Solo afirmamos diferencias con IC que excluye 0 tras la corrección; el resto lo declaramos sin decidir."))
# 2. HPO desigual
obj.append(("DistilBERT tuvo una búsqueda de hiperparámetros mucho menor que los modelos ligeros: la comparación de H4 no es justa.",
            "Es cierto y lo declaramos en el pre-registro: 3 configuraciones con una semilla frente a 30 *trials* en la mayoría de los modelos "
            "ligeros (la BiLSTM agotó su tope de tiempo antes; el capítulo {doc}`22_nlp_modelos` da el número real). El sesgo va en contra de DistilBERT, "
            "de modo que una ventaja suya sería conservadora; una desventaja podría reducirse con más búsqueda. No podemos descartar esa segunda "
            "posibilidad con los datos actuales."))
# 3. atajo de categoría
pt = NL["perturbations"]
d = pt[(pt.perturbation == "sin_categoria") & (pt.model == served)]
obj.append(("El clasificador de texto puede estar leyendo el nombre de la categoría en el propio currículo, no entendiendo el documento.",
            (f"Lo medimos: al borrar las palabras de la categoría, la F1 macro de {LN.get(served, served)} cae {d["drop"].iloc[0]:.3f} "
             f"[{d.drop_lo.iloc[0]:.3f}, {d.drop_hi.iloc[0]:.3f}]. " if len(d) else "") +
            "La cifra de la tabla principal incluye esa pista y sobreestima el desempeño en currículos que no nombran su categoría; lo declaramos en la "
            "tarjeta del modelo."))
# 4. anotaciones de segmentación
obj.append(("Las anotaciones de Sartorius tienen errores: el AP mide acuerdo con anotaciones imperfectas, no calidad de segmentación.",
            "Es una amenaza de constructo que confirmamos en la inspección visual del capítulo de EDA de segmentación. No reanotamos; la taxonomía de "
            "errores aísla las detecciones sin célula de referencia («falsas»), que es donde se concentra el efecto de las células sin anotar. "
            "Las comparaciones entre modelos son válidas porque todos se miden contra las mismas anotaciones; el valor absoluto del AP no lo es."))
# 5. equidad
cf = NL["fairness"]["counterfactual"]
r = cf[cf.model == served]
_sgn = NL["fairness"]["subgroups"].groupby("group").n.first()
_n_g = 373 - int(_sgn.get("sin términos de género", 0))
obj.append(("El análisis de equidad usa términos de género del texto como indicio; eso no mide discriminación real.",
            (f"De acuerdo. El contrafactual mide algo más limitado pero bien definido: si la predicción cambia al cambiar el género gramatical "
             f"(con {LN.get(served, served)} cambia en {int(r.n_flips.iloc[0])} de los {_n_g} currículos de test que contienen algún "
             "término de género; los demás no cambian al reescribirlos). " if len(r) else "") +
            "Las métricas por grupo usan grupos pequeños e intervalos anchos. Por eso concluimos que el modelo no debe usarse para decidir sobre "
            "personas, y que una auditoría real requiere atributos declarados voluntariamente."))
for k, (o, resp) in enumerate(obj, 1):
    md(f"**Objeción {k}.** {o}\n\n*Respuesta.* {resp}")

**Objeción 1.** El test es pequeño: las diferencias entre modelos podrían ser ruido de muestreo.

*Respuesta.* Es una limitación real. Segmentación usa 93 imágenes de test y texto 373 documentos (≈ 15 por clase). Por eso reportamos IC *bootstrap* en todas las cifras, corrección de Holm en todas las familias de comparaciones y, en texto, la *t* de Nadeau–Bengio sobre 5 pliegues además de McNemar. Solo afirmamos diferencias con IC que excluye 0 tras la corrección; el resto lo declaramos sin decidir.

**Objeción 2.** DistilBERT tuvo una búsqueda de hiperparámetros mucho menor que los modelos ligeros: la comparación de H4 no es justa.

*Respuesta.* Es cierto y lo declaramos en el pre-registro: 3 configuraciones con una semilla frente a 30 *trials* en la mayoría de los modelos ligeros (la BiLSTM agotó su tope de tiempo antes; el capítulo {doc}`22_nlp_modelos` da el número real). El sesgo va en contra de DistilBERT, de modo que una ventaja suya sería conservadora; una desventaja podría reducirse con más búsqueda. No podemos descartar esa segunda posibilidad con los datos actuales.

**Objeción 3.** El clasificador de texto puede estar leyendo el nombre de la categoría en el propio currículo, no entendiendo el documento.

*Respuesta.* Lo medimos: al borrar las palabras de la categoría, la F1 macro de DistilBERT cae 0.604 [0.569, 0.641]. La cifra de la tabla principal incluye esa pista y sobreestima el desempeño en currículos que no nombran su categoría; lo declaramos en la tarjeta del modelo.

**Objeción 4.** Las anotaciones de Sartorius tienen errores: el AP mide acuerdo con anotaciones imperfectas, no calidad de segmentación.

*Respuesta.* Es una amenaza de constructo que confirmamos en la inspección visual del capítulo de EDA de segmentación. No reanotamos; la taxonomía de errores aísla las detecciones sin célula de referencia («falsas»), que es donde se concentra el efecto de las células sin anotar. Las comparaciones entre modelos son válidas porque todos se miden contra las mismas anotaciones; el valor absoluto del AP no lo es.

**Objeción 5.** El análisis de equidad usa términos de género del texto como indicio; eso no mide discriminación real.

*Respuesta.* De acuerdo. El contrafactual mide algo más limitado pero bien definido: si la predicción cambia al cambiar el género gramatical (con DistilBERT cambia en 0 de los 95 currículos de test que contienen algún término de género; los demás no cambian al reescribirlos). Las métricas por grupo usan grupos pequeños e intervalos anchos. Por eso concluimos que el modelo no debe usarse para decidir sobre personas, y que una auditoría real requiere atributos declarados voluntariamente.